# 01 · Data Understanding
Explore the Kaggle IBM credit card customer data. This notebook does not modify source data or train a model. Run with the working directory set to `notebooks/`.

If the CSV is absent, download `BankChurners.csv` from https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers and place it in `data/raw/`. See `../data/README.md`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid')

In [ ]:
data_path = Path('../data/raw/BankChurners.csv')
if not data_path.exists(): raise FileNotFoundError(f'{data_path} missing; follow ../data/README.md')
df = pd.read_csv(data_path)
print('Dimensions:', df.shape)
display(df.head())
display(df.tail())
print('Columns:', df.columns.tolist())
df.info()
display(df.describe())
display(df.describe(include='object'))

## Target variable: Attrition_Flag
Historical churn and retention rates describe this sample, not future predictions.

In [ ]:
target = 'Attrition_Flag'
counts = df[target].value_counts(dropna=False)
total = len(df)
existing = int(counts.get('Existing Customer', 0))
attrited = int(counts.get('Attrited Customer', 0))
print(f'Existing: {existing:,}; attrited: {attrited:,}; total: {total:,}')
print(f'Churn: {attrited/total*100:.2f}%; retention: {existing/total*100:.2f}%')
display(pd.DataFrame({'count': counts, 'percent': counts/total*100}))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x=target, order=counts.index, ax=ax[0])
ax[0].tick_params(axis='x', rotation=15); ax[0].set_title('Customer count')
ax[1].bar(counts.index.astype(str), counts/total*100); ax[1].set_title('Customer distribution (%)')
ax[1].tick_params(axis='x', rotation=15); plt.tight_layout()

## Data quality
Range rules are plausibility flags for review, not automatic corrections. Confirm unusual values against source documentation.

In [ ]:
print('Missing by column:'); display(df.isna().sum().to_frame('missing').query('missing > 0'))
print('Total missing:', int(df.isna().sum().sum()))
print('Exact duplicate rows:', int(df.duplicated().sum()))
if 'CLIENTNUM' in df: print('Duplicate CLIENTNUM:', int(df.CLIENTNUM.duplicated().sum()))
cat_cols = df.select_dtypes(include=['object','category']).columns
for c in cat_cols: print(c, df[c].value_counts(dropna=False).to_dict())
nums = df.select_dtypes(include=np.number)
display(pd.DataFrame({'min':nums.min(), 'max':nums.max(), 'unique':nums.nunique()}))
rules = {'Customer_Age outside 18-120': ('Customer_Age', lambda s: ~s.between(18,120)), 'Avg_Utilization_Ratio outside 0-1': ('Avg_Utilization_Ratio', lambda s: ~s.between(0,1)), 'Credit_Limit <= 0': ('Credit_Limit', lambda s: s<=0)}
flags = {name: int(fn(df[col]).fillna(False).sum()) for name,(col,fn) in rules.items() if col in df}
display(pd.Series(flags, name='flagged rows').to_frame())
display(pd.DataFrame({'check':['rows','columns','missing cells','exact duplicate rows'], 'result':[len(df),df.shape[1],int(df.isna().sum().sum()),int(df.duplicated().sum())]}))

## Business understanding
- **Demographics:** `Customer_Age`, `Gender`, `Dependent_count`, `Education_Level`, `Marital_Status` support customer segmentation; apply policy and fairness review before operational use.
- **Card relationship and usage:** `Card_Category`, `Months_on_book`, `Credit_Limit`, `Total_Revolving_Bal`, `Avg_Open_To_Buy` describe product, tenure, and available/revolving credit.
- **Transaction activity:** `Total_Trans_Amt` and `Total_Trans_Ct` summarize amount and count; `Total_Amt_Chng_Q4_Q1` and `Total_Ct_Chng_Q4_Q1` compare quarter activity. Values below one indicate a decline from the earlier quarter.
- **Inactivity and contact:** `Months_Inactive_12_mon` counts inactive months in the prior year; `Contacts_Count_12_mon` counts bank contacts. These may signal engagement or service needs, not prove cause.
- **Credit utilization:** `Avg_Utilization_Ratio` describes revolving balance relative to available credit, normally 0–1. Interpret with balances and limits.
- **Attrition:** `Attrition_Flag` is the historical outcome (`Existing Customer` or `Attrited Customer`). `CLIENTNUM` is an identifier. The `Naive_Bayes_Classifier*` fields are precomputed model outputs and must not be future predictors because of target leakage.